# LLM Evaluation

Build a small DGIdb/PubMed answer set, optionally run Wags-LLM extraction, and score the predictions.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from answer_sheet import (
    build_answer_sheet,
    task_payloads_from_pmids,
    task_pmids_from_abstracts,
)
from dgidb import filter_eligible_interactions, load_cached_dgidb_source
from pubmed import fetch_pubmed_abstracts, filter_interactions_with_abstracts
from scoring import score_predictions
from utils import out_path
from wags import prediction_status_frame, run_wags_predictions, save_predictions

In [ ]:
ENABLE_LLM_CALLS = False
N_PMIDS = 30
RANDOM_SEED = 42
REQUIRE_PMID_SPECIFIC_INTERACTION_TYPE = True
MECHANISTIC_TYPES = ("inhibitor", "activator")

OUT_DIR = Path("outputs")
CACHE_DIR = OUT_DIR / "dgidb_cache"
OUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DGIDB_URL = "https://www.dgidb.org/data/latest/interactions.tsv"
DGIDB_GRAPHQL_URL = "https://dgidb.org/api/graphql"
PUBMED_EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

MODEL_KEY = "claude-sonnet-4-6"
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
AWS_PROFILE_NAME = "dev-account"
BEDROCK_REGION = "us-east-2"
PROMPT_NAME = "dgidb_pubmed_inhibitor_activator_extraction"
PROMPT_VERSION = "v2026-06-16"
PROMPT_VERSION_SLUG = PROMPT_VERSION.replace("/", "_").replace(":", "_")

PATHS = {
    "dgidb_summary_cache": CACHE_DIR / "interactions.tsv",
    "dgidb_source_cache": CACHE_DIR / "publication_interactions.csv",
    "pubmed_abstract_cache": OUT_DIR / "pubmed_abstracts.csv",
    "answer_sheet": out_path(OUT_DIR, "answer_sheet", n_pmids=N_PMIDS),
    "pmid_tasks": out_path(OUT_DIR, "pmid_tasks_with_abstracts", n_pmids=N_PMIDS),
    "prompt_payloads": out_path(
        OUT_DIR, "wags_prompt_payloads", suffix="json", n_pmids=N_PMIDS
    ),
    "predictions": OUT_DIR
    / f"predictions_{MODEL_KEY}_{PROMPT_VERSION_SLUG}_{N_PMIDS}.json",
    "metrics": OUT_DIR / f"metrics_{MODEL_KEY}_{PROMPT_VERSION_SLUG}_{N_PMIDS}.csv",
}
PATHS

## DGIdb candidates

In [ ]:
source_df = load_cached_dgidb_source(
    DGIDB_URL,
    summary_cache_path=PATHS["dgidb_summary_cache"],
    source_cache_path=PATHS["dgidb_source_cache"],
    graphql_url=DGIDB_GRAPHQL_URL,
    mechanistic_types=MECHANISTIC_TYPES,
)

eligible_df = filter_eligible_interactions(
    source_df,
    require_pmid_specific_interaction_type=REQUIRE_PMID_SPECIFIC_INTERACTION_TYPE,
    mechanistic_types=MECHANISTIC_TYPES,
)
eligible_df.head()

## PubMed abstracts

In [ ]:
eligible_abstracts_df = fetch_pubmed_abstracts(
    sorted(eligible_df["pmid"].astype(str).unique()),
    cache_path=PATHS["pubmed_abstract_cache"],
    efetch_url=PUBMED_EFETCH_URL,
)

eligible_df, eligible_abstracts_df, abstract_filter_stats = (
    filter_interactions_with_abstracts(
        eligible_df,
        eligible_abstracts_df,
        min_pmids=N_PMIDS,
    )
)
pd.DataFrame([abstract_filter_stats])

## Answer sheet and task payloads

In [ ]:
answer_sheet, answer_sheet_full, selected_pmids = build_answer_sheet(
    eligible_df,
    n_pmids=N_PMIDS,
    seed=RANDOM_SEED,
    require_pmid_specific_interaction_type=REQUIRE_PMID_SPECIFIC_INTERACTION_TYPE,
    mechanistic_types=MECHANISTIC_TYPES,
)

task_pmids = task_pmids_from_abstracts(selected_pmids, eligible_abstracts_df)
task_payloads = task_payloads_from_pmids(task_pmids)

answer_sheet.to_csv(PATHS["answer_sheet"], index=False)
task_pmids.to_csv(PATHS["pmid_tasks"], index=False)
PATHS["prompt_payloads"].write_text(
    json.dumps(task_payloads, indent=2, ensure_ascii=False) + "\n"
)

answer_sheet.head()

## Wags-LLM predictions

In [ ]:
predictions = run_wags_predictions(
    task_payloads,
    predictions_path=PATHS["predictions"],
    enable_llm_calls=ENABLE_LLM_CALLS,
    model_key=MODEL_KEY,
    model_id=MODEL_ID,
    prompt_name=PROMPT_NAME,
    prompt_version=PROMPT_VERSION,
    aws_profile_name=AWS_PROFILE_NAME,
    bedrock_region=BEDROCK_REGION,
    max_tokens=2048,
    temperature=0.0,
    allowed_pmids=task_pmids["pmid"].astype(str),
)
if predictions:
    save_predictions(predictions, PATHS["predictions"])

prediction_status_frame(predictions)

## Score predictions

In [ ]:
metrics_df = score_predictions(
    answer_sheet,
    predictions_path=PATHS["predictions"],
    metrics_path=PATHS["metrics"],
    mechanistic_types=MECHANISTIC_TYPES,
)
metrics_df

## Generated outputs

In [ ]:
pd.DataFrame(
    {"name": name, "path": str(path), "exists": Path(path).exists()}
    for name, path in PATHS.items()
)